# Test: Convert a Minutes Paper to Custom XML Schema

General schema format:
```
<committeeDoc>
<documentType> AP <\documentType>
<committeeName> SEN <\committeeName>
<committeeMeetingData> 2024-05-22 </committeeMeetingData>
<committeeYear> 2023/4 </committeeYear>
<meetingNumber> 3 </meetingNumber>
<items>
<agendaItem number="1" paper = "S 23/24 3A">
<metadata> ... </metadata>
<content> ... </content>
<\agendaItem>
     ...
     ...
<\items>
<\committeeDoc>
```

## Imports and file definitions

In [1]:
import pymupdf
from lxml import etree
import re
import pandas as pd
import base64

doc = pymupdf.open("SEN_M_20240618.pdf")
doc_with_table = pymupdf.open("SEN_AP_20240522.pdf")
temp_filepath = "SenatePapers23-24.csv"

## Understand how blocks are automatically detected

In [2]:
page = doc[0]
blocks = page.get_text("dict", sort=True)["blocks"]
for block in blocks:
    if block["type"] == 0:
        # Get the raw text with newlines visible
        spans = [span for line in block["lines"] for span in line["spans"]]
        raw_text = " ".join(s["text"] for s in spans)
        
        print(f"=== BLOCK ===")
        print(f"Raw text repr: {repr(raw_text)}")
        print(f"Num lines: {len(block['lines'])}")
        print(f"Font size: {spans[0]['size'] if spans else 'N/A'}")
        print(f"Bold: {bool(spans[0]['flags'] & 16) if spans else 'N/A'}")
        print()

=== BLOCK ===
Raw text repr: '  Senatus Academicus  Reconvened Meeting '
Num lines: 3
Font size: 10.979999542236328
Bold: True

=== BLOCK ===
Raw text repr: '  Tuesday 18 June 2024 at 9:45-10:45am '
Num lines: 2
Font size: 10.979999542236328
Bold: False

=== BLOCK ===
Raw text repr: 'Microsoft Teams   '
Num lines: 2
Font size: 10.979999542236328
Bold: False

=== BLOCK ===
Raw text repr: 'Confirmed Minute    Attendees:  Peter Adkins, Gill Aitken, Mteeve Amugune, Arianna Andreangeli, Jonathan  Ansell, Kate Ash-Irisarri, Michael Barany, Laura Bickerton, Richard Blythe, Catherine Bovill,  Holly Branigan, Aidan Brown, Rory Callison, Jeremy Carrette, Leigh Chalmers, Neil Chue  Hong, Juan Cruz, Sarah Cunningham-Burley, Sumari Dancer, Luigi Del Debbio, Chris Dent,  Charlotte Desvages, Simone Dimartino, Claire Duncanson, Murray Earle, Tonks Fawcett,  Samantha Fawkner, Manuel Fernandez-Gotz, Chris French, Vashti Galpin, Soledad Garcia  Ferrari, Benjamin Goddard, Richard Gratwick, Colm Harmon, Ga

## Test table extraction

In [3]:
page_with_table = doc_with_table[113]
table_blocks = page_with_table.get_text("dict", sort=True)["blocks"]
tables = page_with_table.find_tables()
if tables:
    # 3. Extract the first table
    my_table = tables[0]
    
    # 4. Convert to a readable Pandas DataFrame
    df = my_table.to_pandas()
    print(df)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
                                                Col0 Area of focus  Col2  \
0  Knowledge and\nengagement –\nunderstanding the...          None  None   
1  Research\ngrants/projects –\ncontinuous\nimpro...          None  None   

                                                Col3 Where we are now  Col5  \
0  Key processes and\ndocumentation are\ndifferen...             None  None   
1  Projects not set up\nquickly enough and are\nm...             None  None   

                                                Col6 Where we want to be  \
0  Business processes and\noperating procedures t...                None   
1  We have processes in place\nwhich are simple, ...                None   

   Col8                                               Col9 What we’re doing  \
0  None  • Publish standard operating procedures in\na ...             None   
1  None  • Assess and tackle backlogs to support\nset-u... 

## Create custom blocks

In [4]:
# Returns true if a block overlaps with any of the tables on a page
def overlaps_table(block_bbox, table_bboxes):
    bx0, by0, bx1, by1 = block_bbox
    for tx0, ty0, tx1, ty1 in table_bboxes:
        # If the block is entirely outside the table in any direction, no overlap
        if bx1 < tx0 or bx0 > tx1 or by1 < ty0 or by0 > ty1:
            continue
        return True
    return False

# Returns true if a block overlaps with any of the images on a page
def overlaps_img(block_bbox, img_bboxes):
    bx0, by0, bx1, by1 = block_bbox
    for ix0, iy0, ix1, iy1 in img_bboxes:
        if bx1 < ix0 or bx0 > ix1 or by1 < iy0 or by0 > iy1:
            continue
        return True
    return False  

# Returns all hyperlinks for a page along with the link's bbox
def get_page_links(page):
    links = []
    for link in page.get_links():
        if link.get("uri"):  # skip internal page jumps
            links.append({
                "bbox": link["from"], 
                "uri": link["uri"],
            })
    return links

# Returns a dictionary of text spans w/ the matching link
def find_links_in_spans(line_spans, page_links):
    found_links = []
    for span in line_spans:
        sx0, sy0, sx1, sy1 = span["bbox"]
        for link in page_links:
            lx0, ly0, lx1, ly1 = link["bbox"]
            # Check for matching bbox (w/ 1.5 pt tolerance)
            if sx0 >= lx0 - 1.5 and sy0 >= ly0 - 1.5 and sx1 <= lx1 + 1.5 and sy1 <= ly1 + 1.5:
                found_links.append({
                    "text": span["text"], 
                    "uri": link["uri"],    
                })
    return found_links

# Joins all accumulated lines into one block dict
def make_sub_block(current_lines):
    text = " ".join(l["text"] for l in current_lines) 
    text = re.sub(r'\s+', ' ', text).strip() # clean up by getting rid of extra whitespace
    if not text:
        return None  # caller should check for None and skip appending
    return {
        "text": text,
        "font_size": current_lines[0]["font_size"],
        "font": current_lines[0]["font"],
        "bold": current_lines[0]["bold"],
        "bbox": current_lines[0]["bbox"],
        "type": "text",
        "links": [link for l in current_lines for link in l.get("links", [])] # use l.get("links", []) to prevent key error on lines w/o links
    }

# Returns whether a text block is a page number
def is_page_num(block):
    text = block["text"].strip()
    if block["type"] != "text":
        return False
    # Matches page numbers: "1", "12", etc.
    if re.fullmatch(r'\d+', text):
        return True
     # Matches "Page 1 of 9", "Page 12 of 100", etc. 
    if re.fullmatch(r'[Pp]age\s+\d+\s+of\s+\d+', text):
        return True
    return False

# Seperates bullet points in a block
def clean_bullet_points(text):
    text = re.sub(r'\s*•', '\n•', text)
    text = re.sub(r'\s*(?<!\w)o(?!\w)', '\n•', text)
    return text.strip()

# Extract blocks across all pages of a document
def extract_blocks(doc):
    all_blocks = []
    for page in doc:
        page_blocks = []
        page_links = get_page_links(page)
        
        # Extract tables on page and add each cell as a block
        table_bboxes = []
        for table in page.find_tables():
            table_bboxes.append(table.bbox)
            
            rows = table.extract()
            for row_i, row in enumerate(rows):
                for col_i, cell_text in enumerate(row):
                    if not cell_text or not cell_text.strip(): # skip empty cells
                        continue
                    else:
                        page_blocks.append({
                            "text": re.sub(r'\s+', ' ', cell_text).strip(), # get rid of extra whitespace
                            "type": "table",      
                            "row": row_i,      
                            "col": col_i,       
                            "bbox": table.bbox, # bbox for the whole table 
                        })
                        
        # Extract images and add each image as a block
        img_bboxes = []
        for img in page.get_images():
            xref = img[0]
            
            rects = page.get_image_rects(xref) # gets all bboxes where that image occurs
            if not rects:
                continue
                
            # Take first instance of image as bbox (there should only be 1)
            bbox = rects[0] 
            img_bboxes.append(bbox)
            
            # Extract the image's raw data 
            raw_img = doc.extract_image(xref)
            if raw_img:
                img_bytes = raw_img["image"]  # Raw binary data
                img_ext = raw_img["ext"]      # e.g., 'png', 'jpeg'
                
                # Convert the binary bytes into a Base64 string -> image will live entirely in the XML
                base64_encoded = base64.b64encode(img_bytes).decode("utf-8")
                
                page_blocks.append({
                    "type": "img",
                    "bbox": list(bbox), # need to convert rect object to list
                    "image_data": base64_encoded,
                    "image_ext": img_ext
                })
        
        # Extract clean text blocks and add them
        blocks = page.get_text("dict", sort = True)["blocks"]
        for block in blocks:
            # Only look at text blocks (type 0)
            if block["type"] != 0:
                continue

            # Skip block if it overlaps with tables or images (content already captured)
            if overlaps_table(block["bbox"], table_bboxes) or overlaps_img(block["bbox"], img_bboxes):
                continue
            
            # Flatten all spans in the block to ensure the block actually contains text
            spans = [span for line in block["lines"] for span in line["spans"]]
            if not spans: 
                continue
            
            # Build a metadata dictionary per line
            lines_meta = []
            for line in block["lines"]:
                line_spans = line["spans"]
                if not line_spans:
                    continue
                    
                # Combine text across all spans for the line
                line_text = " ".join(span["text"] for span in line_spans)
                
                # Identify the 'dominant' span (the longest text piece) to represent the line's style
                line_dominant = max(line_spans, key=lambda s: len(s["text"])) 
                
                lines_meta.append({
                    "text": line_text,
                    "bbox": line_dominant["bbox"],
                    "font": line_dominant["font"],
                    "font_size": line_dominant["size"],
                    "bold": bool(line_dominant["flags"] & 16), # Bitwise check for bold flag
                    "type": "text",
                    "links": find_links_in_spans(line_spans, page_links) 
                })

            # Group lines into sub-blocks based on formatting continuity
            sub_blocks = []
            current_lines = [] 
            
            for line in lines_meta:
                if not current_lines:
                    current_lines.append(line)
                    continue

                prev_line = current_lines[-1]
                should_split = False

                # Split conditions: change in size, font, bold state, or large whitespace gaps
                if abs(line["font_size"] - prev_line["font_size"]) > 1:
                    should_split = True
                elif line["font"] != prev_line["font"]:
                    should_split = True
                elif line["bold"] != prev_line["bold"]:
                    should_split = True
                elif re.search(r'\n{2,}', line["text"]):
                    should_split = True
                elif re.search(r'\s{3,}', line["text"]):
                    should_split = True

                if should_split:
                    new_block = make_sub_block(current_lines)
                    if new_block is not None:
                        sub_blocks.append(new_block)
                    current_lines = [line] # Reset and start a new sub-block with the current line
                else:
                    current_lines.append(line) # Append to the ongoing sub-block

            # Add the final accumulated group of lines
            if current_lines:
                text = " ".join(l["text"] for l in current_lines)
                text = re.sub(r'\s+', ' ', text).strip()
                if text:
                    new_block = make_sub_block(current_lines)
                    if new_block is not None:
                        sub_blocks.append(new_block)

                # Save valid sub-blocks to the global page blocks list
                for b in sub_blocks:
                    if b["text"]:
                        page_blocks.append(b)
                        
        # Sort blocks in page to appear by position 
        page_blocks.sort(key=lambda b: b["bbox"][1]) # bbox[1] = y0
        
        # Put newlines where there are bulletpoints ("•" or "o")
        for block in page_blocks:
            if block["type"] != "img":
                block["text"] = clean_bullet_points(block["text"])
        
        all_blocks.extend(page_blocks)
                
                
    # Merge blocks that are part of the same paragraph together
    extracted_blocks = []
    buffer = None

    for block in all_blocks:
        # If we encounter an image, clear out the buffer
        if block["type"] == "img":
            if buffer:
                extracted_blocks.append(buffer)
                buffer = None
            extracted_blocks.append(block)
            continue
            
        if is_page_num(block): # skip page nums for XML conversion
            continue
        
        curr_text = block["text"].strip()
        if curr_text == "H/02/02/02": # Unnecessary heading frequently at the top of SEN papers
            continue
            
        # If there's no active buffer, make this block the buffer
        if buffer is None:
            buffer = block
            continue

        # Check for paragraph continuation rules
        lowercase_start = curr_text[0].islower() if curr_text else False
        list_item = re.fullmatch(r'\d+\.', buffer["text"])
        bullet_point = re.fullmatch(r'•', buffer["text"])
        same_paragraph = lowercase_start or list_item or bullet_point
        
        if same_paragraph:
            buffer["text"] = buffer["text"] + " " + curr_text
        else:
            extracted_blocks.append(buffer)
            buffer = block

    if buffer:
        extracted_blocks.append(buffer)
    
    return extracted_blocks

# Print extracted blocks
for block in extract_blocks(doc_with_table[125:128]):
    if block["type"] == "img":
        print("IMAGE GOES HERE")
    else:
        print(block["text"])
    print("---")

S 23/24 3S
---
Senate
---
22 May 2024 Report from Central Academic Promotions Committee
---
Description of paper
---
1. Report of the recommendations of the Central Academic Promotions Committee provided in Appendix 1.
---
Action requested / Recommendation
---
2. For information.
---
Resource implications
---
3. Increased salaries will impact on each individual College’s staff budget.
---
Risk Management
---
4. N/A
---
Responding to the Climate Emergency and Sustainable Development Goals
---
5. N/A
---
Equality and Diversity
---
6. Equality and Diversity is central to the considerations of the Central Academic Promotions Committee.
---
Communication, implementation and evaluation of the impact of any action agreed
---
7. N/A
---
Further information
---
Presenter(s) (if required)
---
Author(s)
---
Shanthi Menon HR Partner Reward University HR 13 May 2024
---
Freedom of information: Open
---
Appendix 1:
---
Title
---
Initial
---
Surname
---
School/Deanery
---
Personal Chair Title
---
Dat

## Extracting metadata
The following code extracts metadata from a CSV file. If the file is not found within the CSV by its filename, then it is instead extracted automatically from the filename. However, as meeting number cannot be inferred from file name, it is left as `Unknown`.

Metadata:
* Document type
* Committee name
* Meeting date (or meeting start date and end date)
* Committee year
* Meeting number

Notes:
* Another solution is to have start date be the same as end date for single day meetings
* Meeting number cannot automatically be extracted for minutes. For agenda and papers, it can be deduced from the paper codes. But minutes do not generally include paper codes (except in reference to past/future meetings), which would be risky to extract incorrectly
* We are currently assuming that August 1st begins a new academic year

In [5]:
# Extract metadata automatically from CSV
def fetch_metadata_from_csv(target_file, csv_path):
    df = pd.read_csv(csv_path)
    match = df[df['fileName'] == target_file] # no ext (such as .pdf)
    if match.empty:
        return None
    else:
        return match.iloc[0].to_dict() 

# Extract metadata automatically from filename (does not extract meeting number)
def extract_metadata_from_filename(target_file):
    parts = target_file.split('_')
    e_meeting = len(parts) > 3
    
    # Extract doc type
    extracted_doc_type = parts[1]

    # Extract committee name
    extracted_committee_name = None
    if parts[0] == "SEN": #in CSV, Senate and E-Senate do not keep abbreviation
        if e_meeting:
            extracted_committee_name = "E-Senate"
        else:
            extracted_committee_name = "Senate"
    else:
        extracted_committee_name = parts[0]
        
    # Extract meeting date(s)
    extracted_date = None
    extracted_start_date = None
    extracted_end_date = None

    if e_meeting:
        extracted_start_date = parts[2]
        extracted_end_date = parts[3]
    else:
        extracted_date = parts[2]
        
    def fix_date_formatting(d): # DD/MM/YYYY
        if d:
            return f"{d[6:]}/{d[4:6]}/{d[:4]}"
        return None
    extracted_date, extracted_start_date, extracted_end_date = map(fix_date_formatting, (extracted_date, extracted_start_date, extracted_end_date))
    
    extracted_dates = None
    if e_meeting:
        extracted_dates = f"{extracted_start_date} to {extracted_end_date}"
    else:
        extracted_dates = extracted_date
        
    # Extract academic year
    extracted_academic_year = None
    def extract_year(d):
        year = int(d[:4])
        new_academic_year = f"{year}0801" #August 1 of the year
        if d >= new_academic_year: 
            return f"{year}/{year+1}"
        return f"{year-1}/{year}"
    if e_meeting:
        extracted_academic_year = extract_year(extracted_start_date)
    else:
        extracted_academic_year = extract_year(extracted_date)
    
    return {
        "fileName": target_file,
        "documentType": extracted_doc_type,
        "committeeName": extracted_committee_name,
        "dates": extracted_dates,
        "academicYear": extracted_academic_year,
        "meetingNumber": "Unknown"
    }

# Extract metadata
def extract_metadata(file, csv_filepath):
    filename = file.name.removesuffix('.pdf')
    
    # Validate filename
    valid_meeting_pattern = r"^(SEN|SEC|APRC|SQAC)_(AP|M)_20\d{2}(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])$" #supports years 2000-2099
    valid_e_meeting_pattern = r"^(SEN|SEC|APRC|SQAC)_(AP|M)_20\d{2}(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])_20\d{2}(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])_e$"
    if not(re.match(valid_meeting_pattern, filename) or re.match(valid_e_meeting_pattern, filename)):
        raise Exception("Invalid filename format")
    
    # Try extraction from CSV first
    metadata = fetch_metadata_from_csv(filename, csv_filepath)
    
    # Fallback to data extraction from filename
    if metadata is None:
        metadata = extract_metadata_from_filename(filename)
        
    return metadata
    
print(extract_metadata(doc, temp_filepath))


{'fileName': 'SEN_M_20240618', 'documentType': 'M', 'committeeName': 'Senate', 'dates': '18/6/2024', 'academicYear': '2023/2024', 'meetingNumber': 4}


## Create XML document
Note:
* Including just 1 tag for date; if over multiple days, included in the following format: "2024-06-10 to 2024-06-18"

In [ ]:
# Creates a <tag> element under parent_el
# Embeds links as <a> inline
def build_text_el(parent_el, tag, block):
    el = etree.SubElement(parent_el, tag)
    
    # No links - set text normally
    if not block.get("links"):
        el.text = block["text"]
        return el
    
    remaining_text = block["text"]
    for link in block["links"]:
        link_text = link["text"]
        uri = link["uri"]
        
        # Find where the linked text sits within the remaining text
        link_i = remaining_text.find(link_text)
        if link_i == -1: # text not found
            continue 
        
        before = remaining_text[:link_i]
        after = remaining_text[link_i + len(link_text):]  
        
        # Text before the link
        if len(el) == 0:
            el.text = (el.text or "") + before # or "" handles when el.text = None
        else:
            el[-1].tail = (el[-1].tail or "") + before # el[-1] is the last <a> tag
        
        # Create the inline <a> element for this link
        a = etree.SubElement(el, "a")
        a.set("href", uri)
        a.text = link_text
        
        remaining_text = after 
    
    # Text left after the last link
    if len(el) > 0:
        el[-1].tail = (el[-1].tail or "") + remaining_text 
    else: # edge case of link extraction not working properly -> none of the links found within text
        el.text = (el.text or "") + remaining_text
    
    return el

# Converts pdf file to xml file 
def convert_pdf_to_xml(file, csv_filepath):
    root = etree.Element("committeeDoc")
    tree = etree.ElementTree(root)
    metadata = extract_metadata(file, csv_filepath)

    # Connect to the XSLT styling sheet
    xslt = etree.ProcessingInstruction(
        "xml-stylesheet", 'type="text/xsl" href="style.xslt"'
    )
    root.addprevious(xslt)

    # Add metadata (docType, committeeName, dates, academicYear)
    metadata_el = etree.SubElement(root, "metadata")
    doc_type = etree.SubElement(metadata_el, "documentType")
    doc_type.text = metadata["documentType"]
    committee_name = etree.SubElement(metadata_el, "committeeName")
    committee_name.text = metadata["committeeName"]
    dates = etree.SubElement(metadata_el, "dates")
    dates.text = metadata["dates"]
    academic_year = etree.SubElement(metadata_el, "academicYear")
    academic_year.text = metadata["academicYear"]

    # Loop through extracted blocks
    # Current identification --> bold = heading, anything else = bodyText
    current_item = root
    current_table = None
    current_row = None
    current_row_i = -1
    for block in extract_blocks(file):
        type = block["type"]
        
        if type == "img":
            img_el = etree.SubElement(current_item, "image", ext = block["image_ext"])
            img_el.text = block["image_data"]
        
        elif type == "table":
            text = block["text"]
            
            # Create a new table element
            if current_table is None:
                current_table = etree.SubElement(current_item, "table")
                current_row_i = -1  # reset so the first row gets created below

            # Create a new row when the row index changes
            if block["row"] != current_row_i:
                current_row = etree.SubElement(current_table, "row")
                current_row_i = block["row"]

            # Append a cell
            cell = etree.SubElement(current_row, "cell")
            cell.text = text
        
        elif type == "text":
            text = block["text"]
            font_size = block["font_size"]
            font = block["font"]
            bold = block["bold"]
            
            # Reset table
            current_table = None
            current_row = None
            current_row_i = -1
            
            # Indicates heading
            if bold:
                build_text_el(root, "boldText", block)
            else:
                build_text_el(root, "bodyText", block)   

    # Save to same directory
    with open("output.xml", "wb") as f:
        tree.write(f, xml_declaration=True, encoding="utf-8", pretty_print=True)
        
    # For now just return blank so it doesn't print a huge XML
    return #etree.tostring(root, pretty_print=True, encoding='utf-8').decode('utf-8') 

# View tree so far
convert_pdf_to_xml(doc_with_table, temp_filepath)

# DEBUGGING: Link extraction not working properly
Conclusion: the tolerance on `find_links_in_spans` was too low which caused the program to miss links that occured within a span

In [9]:
# Does it identify the link? -> YES
print("LINK ON PAGE 4")
page_with_link = doc_with_table[3]
links = get_page_links(page_with_link)
print(links)
print()

# Is the correct text in the same bbox? -> YES
print("TEXT IN LINK'S BBOX")
target_bbox = (92.25, 502.06298828125, 279.60400390625, 514.7119750976562) #same as link bbox
text = page_with_link.get_text("text", clip=target_bbox)
print(text)
print()

# Does find_links_in_spans detect associate the link with the right span? -> NO
blocks = page_with_link.get_text("dict", sort=True)["blocks"]
for block in blocks:
    if block["type"] == 0:
        lines_meta = []
        for line in block["lines"]:
            line_spans = line["spans"]
            if not line_spans:
                continue
            
            # What is the actual bbox of the correct text?
            for span in line["spans"]:
                span_text = span["text"]
                span_bbox = span["bbox"]
                
                if span_text == "Senate agendas, papers and minutes":
                    print("BBOX OF 'Senate agendas, papers and minutes'")
                    print(f"text: {span_text}, bbox: {span_bbox}")
                    print()
            
            if len(find_links_in_spans(line_spans, links)) > 0:   
                print("WITH TOLERANCE INCREASED FROM 1 PT TO 1.5")
                print(find_links_in_spans(line_spans, links))

LINK ON PAGE 4
[{'bbox': Rect(92.25, 502.06298828125, 279.60400390625, 514.7119750976562), 'uri': 'https://www.ed.ac.uk/academic-services/committees/senate/agendas-papers'}]

TEXT IN LINK'S BBOX
Senate agendas, papers and minutes.


BBOX OF 'Senate agendas, papers and minutes'
text: Senate agendas, papers and minutes, bbox: (94.54296875, 501.009033203125, 277.33575439453125, 515.9967041015625)

WITH TOLERANCE INCREASED FROM 1 PT TO 1.5
[{'text': 'Senate agendas, papers and minutes', 'uri': 'https://www.ed.ac.uk/academic-services/committees/senate/agendas-papers'}]
